In [3]:
import pandas as pd
import pydicom
import os
from datetime import datetime

contador = 0

# Obtener la fecha y hora actual
fecha_hora_actual = datetime.now()
fecha_actual = str(fecha_hora_actual.date()) + "_" + str(fecha_hora_actual.hour) + str(fecha_hora_actual.minute)
print("Fecha actual:", fecha_actual)

# Define the tags
tagEdad = (0x0010, 0x1010)
tagkVp = (0x0018, 0x0060)
tagExpTime = (0x0018, 0x1150)
tagmAs = (0x0018, 0x1152)
tagOrganDose = (0x0040, 0x0316)
tagEntranceDose = (0x0040, 0x8302)
tagStudyDescrip = (0x0008, 0x1030)
tagImageLaterality = (0x0020, 0x0060)  # Corregido el tag de '002,0016' a '0020,0060'
tagViewPosition = (0x0018, 0x5101)
tagThickness = (0x0018, 0x11A0)
tagAnode = (0x0018, 0x1191)
tagFilterMaterial = (0x0018, 0x7050)
tagManufacture = (0x0008, 0x0070)
tagInstitutionName = (0x0008,0x0080)

# Definir funciones
def listar_archivos_en_carpeta(ruta_carpeta):
    nombres_archivos = []
    if os.path.isdir(ruta_carpeta):
        archivos_en_carpeta = os.listdir(ruta_carpeta)
        for archivo in archivos_en_carpeta:
            ruta_absoluta = os.path.join(ruta_carpeta, archivo)
            if os.path.isfile(ruta_absoluta):
                nombres_archivos.append(archivo)
    else:
        print("La ruta proporcionada no es una carpeta.")
    return nombres_archivos

def extract_dicom_metadata(dicom_file):
    global contador
    ds = pydicom.dcmread(dicom_file)
    metadata = []

    # Metadata appending
    metadata.append(nombre_archivo)
    metadata.append(ds[tagStudyDescrip].value)
    metadata.append(ds[tagkVp].value)
    metadata.append(ds[tagExpTime].value)
    metadata.append(ds[tagmAs].value)
    metadata.append(ds[tagOrganDose].value*100)
    metadata.append(ds[tagEntranceDose].value)
    metadata.append(ds[tagViewPosition].value)
    metadata.append(ds[tagThickness].value)
    anode_filter = str(ds[tagAnode].value) + "/" + str(ds[tagFilterMaterial].value)
    metadata.append(anode_filter)
    metadata.append(ds[tagManufacture].value)
    metadata.append(ds[tagInstitutionName].value)

    return metadata

# Ruta de la carpeta con imágenes DICOM
ruta_carpeta = "C:/Users/Daniela G/Desktop/PruebaDicom/MAMOGRAFIA"
nombres_archivos = listar_archivos_en_carpeta(ruta_carpeta)

# Definir las columnas del DataFrame
columnas = ['Archivo', 'Estudio', 'kV', 'Tiempo de Exposición mSec', 'mAs', "Organ Dose [mGy]", 'Entrance Dose [mGy]', 'Proyección', 'Espesor', 'Ánodo/Filtro', 'Fabricante','Institución']

df = pd.DataFrame(columns=columnas)

# Extraer datos de cada archivo DICOM
for nombre_archivo in nombres_archivos:
    print(nombre_archivo)
    dicom_file = os.path.join(ruta_carpeta, nombre_archivo)
    metadata = extract_dicom_metadata(dicom_file)
    print(metadata)
    
    if len(metadata) != 0:
        df.loc[len(df)] = metadata
        print("****************************")
    else:
        print("paciente pediátrico")

print(df)

# Especificar la ruta del archivo Excel
ruta_archivo = "C:/Users/Daniela G/Desktop/PruebaDicom/Datos_" + fecha_actual + ".xlsx"

# Guardar el DataFrame en un archivo Excel
df.to_excel(ruta_archivo, index=False)
print("Datos guardados en el archivo Excel:", ruta_archivo)


Fecha actual: 2024-08-30_959
10000007
['10000007', 'Diagnostico', '26.0', '556', '67', 0.6799999999999999, '1.95', 'RCC', '41.0', 'TUNGSTEN/RHODIUM', 'Philips Medical Systems', 'San Vicente Fundación']
****************************
10000008
['10000008', 'Diagnostico', '28.0', '867', '140', 1.52, '5.21', 'LCC', '52.0', 'TUNGSTEN/RHODIUM', 'Philips Medical Systems', 'San Vicente Fundación']
****************************
10000009
['10000009', 'Diagnostico', '29.0', '562', '85', 0.91, '3.44', 'RMLO', '59.0', 'TUNGSTEN/RHODIUM', 'Philips Medical Systems', 'San Vicente Fundación']
****************************
1000000A
['1000000A', 'Diagnostico', '30.0', '1300', '199', 2.04, '9.34', 'LMLO', '70.0', 'TUNGSTEN/RHODIUM', 'Philips Medical Systems', 'San Vicente Fundación']
****************************
10000013
['10000013', 'Diagnostico', '26.0', '122', '11', 0.12, '0.32', 'RCC', '37.0', 'TUNGSTEN/RHODIUM', 'Philips Medical Systems', 'San Vicente Fundación']
****************************
10000014
['1

In [ ]:
import numpy as np
#print(df)

while True:
    T1 = df['Institución'].unique()

    print("\n","Copie y pegue una opción de las siguientes en el cuadro de ingreso:","\n","\n",T1,"\n")

    Hosp=input("Ingrese lugar de interés:  ")

    if Hosp in T1:
        df1=df[df['Institución']==Hosp]
        print("\n",df1,"\n")
        unique_strings = df1['Proyección'].unique()

        print("Valores de estudio existentes para la localizacion ", Hosp,"\n" )
        print(unique_strings)
        
        while True:
            est=input("Ingrese proyeccion a evaluar:")
            if est in unique_strings:
                df2=df1[df1["Proyección"]==est]
                #if est =="CHEST":
                #    unique_strings = df2['Proyeccion'].unique()
                #    print("Valores de estudio existentes para la localizacion ", "\n" )
                #    print(unique_strings)
                proyeccion=est
                df3=df2[df2["Proyección"]==proyeccion]
                print("Valores para estudio de mamografia con proyeccion especifica ", proyeccion,"\n")
                print("\n",df3,"\n")
                nivelRefe = df3["Entrance Dose [mGy]"].median()
                dosisEntrada = df3["Entrance Dose [mGy]"].mean()
                print("\n","Nivel de Referencia en:",est,":",f"{nivelRefe:.2f}","mGy*cm2","\n","Dosis de entrada promedio para mamografía",est,":",f"{dosisEntrada:.2f}","mGy*cm2","\n")
                print("primer percentil ", np.percentile(df3["Entrance Dose [mGy]"], [25]), "     Segundo percentil ", np.percentile(df3["Entrance Dose [mGy]"], [50]), "     Tercer percentil ", np.percentile(df3["Entrance Dose [mGy]"], [75]) )

                print("\n","Kv de nivel de referencia en:",est,":",df3["kV"].median(),"kV","\n","mAs de nivel de referencia en",est,":",df3["mAs"].median(),"mAs","\n","Kv promedio en:",est,":",df3["kV"].mean(),"kV","\n","mAs promedio en",est,":",df2["mAs"].mean(),"mAs","\n")

#                else:
#                    print("Valores para estudio completo ",est,"\n")
#                    print("\n",df2,"\n")
#                    print("\n","Nivel de Referencia en:",est,":",df2["Producto dosis Área"].median(),"mGy*cm2","\n","Producto dosis-área promedio para estudio",est,":",df2["Producto dosis Área"].mean(),"mGy*cm2","\n")
#                    print("primer percentil ", np.percentile(df2["Producto dosis Área"], [25]), "     Segundo percentil ", np.percentile(df2["Producto dosis Área"], [50]), "     Tercer percentil ", np.percentile(df2["Producto dosis Área"], [75]) )
#
#                    print("\n","Kv de nivel de referencia en:",est,":",df2["kV"].median(),"kV","\n","mAs de nivel de referencia en",est,":",df2["mAs"].median(),"mAs","\n","Kv promedio en:",est,":",df2["kV"].mean(),"kV","\n","mAs promedio en",est,":",df2["mAs"].mean(),"mAs","\n")
            elif est not in a:
                print("\n","proyeccion inválido","\n")
                break

    elif Hosp not in T1:
      print("\n",Hosp,"No pertenece a San Vicente Fundación","\n")
      break


 Copie y pegue una opción de las siguientes en el cuadro de ingreso: 
 
 ['San Vicente Fundación'] 



Ingrese lugar de interés:   San Vicente Fundación



       Archivo      Estudio    kV  Tiempo de Exposición mSec  mAs  \
0    10000007  Diagnostico  26.0                        556   67   
1    10000008  Diagnostico  28.0                        867  140   
2    10000009  Diagnostico  29.0                        562   85   
3    1000000A  Diagnostico  30.0                       1300  199   
4    10000013  Diagnostico  26.0                        122   11   
..        ...          ...   ...                        ...  ...   
118  10000264  Diagnostico  29.0                        566   87   
119  1000026C  Diagnostico  26.0                        555   56   
120  1000026D  Diagnostico  26.0                        556   64   
121  1000026E  Diagnostico  27.0                        559   80   
122  1000026F  Diagnostico  27.0                        559   69   

     Organ Dose [mGy]  Entrance Dose [mGy] Proyección  Espesor  \
0                0.68                 1.95        RCC     41.0   
1                1.52                 5.21       

Ingrese proyeccion a evaluar: RCC


Valores para estudio de mamografia con proyeccion especifica  RCC 


       Archivo      Estudio    kV  Tiempo de Exposición mSec  mAs  \
0    10000007  Diagnostico  26.0                        556   67   
4    10000013  Diagnostico  26.0                        122   11   
5    10000014  Diagnostico  30.0                       1278  200   
12   1000001E  Diagnostico  26.0                        555   90   
20   1000002D  Diagnostico  26.0                        554   53   
34   10000051  Diagnostico  29.0                        604   93   
49   10000089  Diagnostico  26.0                        557   89   
53   10000098  Diagnostico  27.0                        559   68   
57   100000AB  Diagnostico  28.0                        561   74   
63   100000BC  Diagnostico  28.0                        564   72   
69   100000D8  Diagnostico  29.0                        565   71   
84   10000118  Diagnostico  25.0                        558   44   
94   100001FB  Diagnostico  28.0              